# Calibration Plots (Reliability Diagrams) — 154-Class
**Owasu Brown | National University | 2025–2026**

A calibration plot (reliability diagram) answers one question:
> *When the model says it is X% confident, is it actually correct X% of the time?*

A perfectly calibrated model sits on the diagonal. Points **above** the diagonal mean the model
is **under-confident** (it's more accurate than it claims). Points **below** mean **over-confident**.

**Models:** InceptionTime · Random Forest · Logistic Regression · CIF  
**Outputs:** `results/end_point/`

```
OBrown_DIS9300_v2/
├── notebooks/          ← this notebook lives here
├── data/
│   ├── npz/            ← ASL_154_test_cif.npz
│   ├── csv/            ← ASL_154_test_rf.csv
│   └── meta/           ← ASL_154_label_encoder.pkl
└── results/
    ├── inc_154/        ← inc_154_proba_test.npy
    ├── rf_154/         ← rf_154_proba_test.npy
    ├── lr_154/         ← lr_154_proba_test.npy
    ├── cif_154/        ← cif_154_proba_test.npy
    └── end_point/      ← ALL outputs saved here
```

**Run order:** Cell 1 → 2 → 3 → 4 → 5

In [ ]:
# ── Cell 1: Mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Imports & paths (new folder structure) ────────────────────────
import os
import numpy as np
import pandas as pd
import joblib
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import label_binarize
warnings.filterwarnings('ignore')

# ── Root (new structure) ──────────────────────────────────────────────────
ROOT        = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Obrown_Dissertation_NU_25/OBrown_DIS9300_v2')
DATA_META   = os.path.join(ROOT, 'data', 'meta')
DATA_NPZ    = os.path.join(ROOT, 'data', 'npz')
DATA_CSV    = os.path.join(ROOT, 'data', 'csv')
RESULTS_DIR = os.path.join(ROOT, 'results')
OUT_DIR     = os.path.join(RESULTS_DIR, 'end_point')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Label encoder ─────────────────────────────────────────────────────────
LE_PATH = os.path.join(DATA_META, 'ASL_154_label_encoder.pkl')
if not os.path.exists(LE_PATH):
    # Fallback: original project root location
    LE_PATH = os.path.join(ROOT, 'ASL_154_label_encoder.pkl')
le          = joblib.load(LE_PATH)
N_CLASSES   = len(le.classes_)
CLASS_NAMES = list(le.classes_)
print(f'Label encoder loaded — {N_CLASSES} classes')

# ── Tol High-Contrast palette (categorical) ───────────────────────────────
TOL = {
    'InceptionTime':       '#0077BB',
    'Random Forest':       '#EE7733',
    'Logistic Regression': '#009988',
    'CIF':                 '#CC3311',
}
MODEL_ORDER = ['InceptionTime', 'Random Forest', 'Logistic Regression', 'CIF']

# ── Model slots (proba files + predictions) ───────────────────────────────
SLOTS = [
    dict(model='InceptionTime',
         proba_file=os.path.join(RESULTS_DIR, 'inc_154', 'inc_154_proba_test.npy'),
         pred_file =os.path.join(RESULTS_DIR, 'inc_154', 'inc_154_predictions.csv')),
    dict(model='Random Forest',
         proba_file=os.path.join(RESULTS_DIR, 'rf_154',  'rf_154_proba_test.npy'),
         pred_file =os.path.join(RESULTS_DIR, 'rf_154',  'rf_154_predictions.csv')),
    dict(model='Logistic Regression',
         proba_file=os.path.join(RESULTS_DIR, 'lr_154',  'lr_154_proba_test.npy'),
         pred_file =os.path.join(RESULTS_DIR, 'lr_154',  'lr_154_predictions.csv')),
    dict(model='CIF',
         proba_file=os.path.join(RESULTS_DIR, 'cif_154', 'cif_154_proba_test.npy'),
         pred_file =os.path.join(RESULTS_DIR, 'cif_154', 'cif_154_predictions.csv')),
]

print(f'Output dir : {OUT_DIR}')

In [ ]:
# ── Cell 3: Load probas & compute calibration curves ─────────────────────
#
# Calibration for multiclass:
#   1. Binarise y_true using One-vs-Rest (154 binary problems)
#   2. Pool all classes together → one global reliability diagram per model
#   3. Also report macro-averaged Expected Calibration Error (ECE)
#      and mean Brier score across all classes
#
# n_bins = 15  (standard for reliability diagrams; more bins = noisier)

N_BINS = 15

def expected_calibration_error(y_true_bin, y_prob, n_bins=N_BINS):
    """ECE: weighted mean |accuracy - confidence| per bin."""
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    ece         = 0.0
    n_total     = len(y_true_bin)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true_bin[mask].mean()
        conf = y_prob[mask].mean()
        ece += mask.sum() / n_total * abs(acc - conf)
    return ece

results = {}   # model → dict with calibration data

for slot in SLOTS:
    m = slot['model']

    if not os.path.exists(slot['proba_file']):
        print(f'  [SKIP] {m} — proba file not found: {slot["proba_file"]}')
        continue

    y_proba = np.load(slot['proba_file'], allow_pickle=True)   # (n_test, 154)

    # Load y_true
    if os.path.exists(slot['pred_file']):
        df_p   = pd.read_csv(slot['pred_file'])
        y_true = df_p['y_true'].values
    else:
        y_true = np.argmax(y_proba, axis=1)   # fallback — calibration will look perfect (don't trust)
        print(f'  [WARN] {m}: no predictions CSV — y_true inferred from argmax (calibration unreliable)')

    n_cls = y_proba.shape[1]

    # ── One-vs-Rest binarisation ──────────────────────────────────────────
    y_bin = label_binarize(y_true, classes=np.arange(n_cls))   # (n_test, n_cls)

    # ── Pool all classes: flatten to one giant (n_test × n_cls,) array ───
    y_true_flat = y_bin.ravel()        # binary ground truth
    y_prob_flat = y_proba.ravel()      # predicted probability

    # ── Global reliability diagram (pooled) ───────────────────────────────
    frac_pos, mean_pred = calibration_curve(
        y_true_flat, y_prob_flat, n_bins=N_BINS, strategy='uniform'
    )

    # ── ECE (pooled) ──────────────────────────────────────────────────────
    ece_pooled = expected_calibration_error(y_true_flat, y_prob_flat, N_BINS)

    # ── Macro Brier score (mean over classes) ─────────────────────────────
    brier_per_class = [
        brier_score_loss(y_bin[:, ci], y_proba[:, ci])
        for ci in range(n_cls)
        if y_bin[:, ci].sum() > 0
    ]
    macro_brier = np.mean(brier_per_class)

    # ── Per-class ECE distribution (for box plot) ──────────────────────────
    ece_per_class = [
        expected_calibration_error(y_bin[:, ci], y_proba[:, ci], N_BINS)
        for ci in range(n_cls)
        if y_bin[:, ci].sum() > 0
    ]

    results[m] = {
        'frac_pos':      frac_pos,
        'mean_pred':     mean_pred,
        'ece_pooled':    ece_pooled,
        'macro_brier':   macro_brier,
        'ece_per_class': ece_per_class,
        'n_test':        len(y_true),
        'n_classes_present': int(y_bin.sum(axis=0).astype(bool).sum()),
    }
    print(f'  [OK] {m:<25}  ECE={ece_pooled:.4f}  Brier={macro_brier:.4f}')

print(f'\n✅ Loaded {len(results)} models')

In [ ]:
# ── Cell 4: Draw all calibration plots ───────────────────────────────────

def save_fig(fig, name, dpi=180):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'  ✅  Saved: {name}')

available = [m for m in MODEL_ORDER if m in results]

# ══════════════════════════════════════════════════════════════════════════
# PLOT 1 — All four models on one reliability diagram (Tol palette)
# ══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect calibration', zorder=1)
ax.fill_between([0,1],[0,1],[1,1], alpha=0.04, color='green', label='Under-confident region')
ax.fill_between([0,1],[0,0],[0,1], alpha=0.04, color='red',   label='Over-confident region')

for m in available:
    r = results[m]
    ax.plot(r['mean_pred'], r['frac_pos'],
            color=TOL[m], lw=2.5, marker='o', markersize=6,
            label=f"{m}  (ECE={r['ece_pooled']:.4f}, Brier={r['macro_brier']:.4f})")

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Mean predicted probability (confidence)', fontsize=11)
ax.set_ylabel('Fraction of positives (actual accuracy)', fontsize=11)
ax.set_title('154-Class · Calibration Plot (Reliability Diagram)\n'
             'All Four Models — One-vs-Rest, Pooled Across Classes',
             fontsize=12, fontweight='bold', pad=14)
ax.legend(frameon=True, fontsize=9,
          loc='upper left', bbox_to_anchor=(1.02, 1),
          borderaxespad=0, framealpha=0.96,
          edgecolor='#CCCCCC')
ax.grid(alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.text(0.01, 0.01,
         'Note. Calibration computed using One-vs-Rest binarisation across all 154 classes. '
         'ECE = Expected Calibration Error (lower = better). '
         'Brier = macro-averaged Brier score (lower = better). '
         'Diagonal = perfect calibration.',
         fontsize=7.5, color='#555555', ha='left', va='bottom')
plt.tight_layout(rect=[0, 0.04, 0.78, 1])
save_fig(fig, 'CAL1_reliability_diagram_all_models.png')

# ══════════════════════════════════════════════════════════════════════════
# PLOT 2 — Individual reliability diagram per model (2×2 grid)
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle('154-Class · Individual Calibration Plots\n'
             '(One-vs-Rest, pooled across all 154 classes)',
             fontsize=13, fontweight='bold', y=1.01)

for ax, m in zip(axes.flat, MODEL_ORDER):
    ax.plot([0,1],[0,1],'k--',lw=1.5,label='Perfect calibration',zorder=1)
    ax.fill_between([0,1],[0,1],[1,1], alpha=0.06, color='#009988', label='Under-confident')
    ax.fill_between([0,1],[0,0],[0,1], alpha=0.06, color='#CC3311', label='Over-confident')

    if m not in results:
        ax.text(0.5, 0.5, f'{m}\n(result pending)',
                ha='center', va='center', fontsize=12,
                color='#AAAAAA', transform=ax.transAxes)
        ax.set_title(m, fontweight='bold', color='#AAAAAA')
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.grid(alpha=0.2)
        continue

    r   = results[m]
    col = TOL[m]

    # Calibration line
    ax.plot(r['mean_pred'], r['frac_pos'],
            color=col, lw=2.5, marker='o', markersize=7, zorder=5,
            label='Model calibration')

    # Confidence histogram (secondary y-axis — Viridis for continuous density)
    ax2 = ax.twinx()
    ax2.hist(r['mean_pred'], bins=N_BINS, range=(0,1),
             color=col, alpha=0.18, density=True)
    ax2.set_ylabel('Density (confidence dist.)', fontsize=8, color='#666666')
    ax2.tick_params(axis='y', labelcolor='#999999', labelsize=7)
    ax2.spines['top'].set_visible(False)

    # Annotations
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Mean predicted probability', fontsize=9)
    ax.set_ylabel('Fraction of positives', fontsize=9)
    ax.set_title(f'{m}', fontsize=11, fontweight='bold', color=col)
    ax.grid(alpha=0.25)
    ax.spines['top'].set_visible(False)

    stats_text = (f"ECE    = {r['ece_pooled']:.4f}\n"
                  f"Brier  = {r['macro_brier']:.4f}\n"
                  f"n_test = {r['n_test']}\n"
                  f"Classes present = {r['n_classes_present']}/{N_CLASSES}")
    ax.text(0.97, 0.04, stats_text, transform=ax.transAxes,
            fontsize=8, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#CCCCCC', alpha=0.9))
    ax.legend(loc='upper left', frameon=True, fontsize=8,
              framealpha=0.9, edgecolor='#CCCCCC')

plt.tight_layout()
save_fig(fig, 'CAL2_reliability_diagram_individual.png')

# ══════════════════════════════════════════════════════════════════════════
# PLOT 3 — ECE & Brier score comparison bar chart (Tol palette)
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('154-Class · Calibration Quality Metrics',
             fontsize=13, fontweight='bold')

for ax, (metric_key, title, ylabel) in zip(axes, [
    ('ece_pooled',  'Expected Calibration Error (ECE)', 'ECE (lower = better calibrated)'),
    ('macro_brier', 'Macro Brier Score',                'Brier score (lower = better)'),
]):
    vals   = [(m, results[m][metric_key]) for m in available]
    vals_s = sorted(vals, key=lambda x: x[1])   # best (lowest) first
    names  = [v[0] for v in vals_s]
    scores = [v[1] for v in vals_s]
    colors = [TOL[m] for m in names]

    bars = ax.barh(names, scores, color=colors, height=0.5,
                   edgecolor='white', linewidth=0.8)
    for bar, (m, v) in zip(bars, vals_s):
        ax.text(v + max(scores)*0.015,
                bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=10,
                fontweight='bold', color=TOL[m])
    ax.set_xlabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim(0, max(scores)*1.30)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()   # best at top
    # Annotate best
    ax.text(scores[0] + max(scores)*0.015,
            bars[0].get_y() + bars[0].get_height()/2 + 0.38,
            '← best', fontsize=8, color='#555555')

plt.tight_layout()
save_fig(fig, 'CAL3_ece_brier_comparison.png')

# ══════════════════════════════════════════════════════════════════════════
# PLOT 4 — Per-class ECE distribution box plot (Viridis — continuous)
# Shows spread of calibration quality across all 154 individual classes
# ══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5.5))
box_data  = [results[m]['ece_per_class'] for m in available]
box_names = available

# Viridis gradient per box (mean ECE drives colour — continuous data)
means = [np.mean(d) for d in box_data]
norm_v = plt.Normalize(min(means), max(means))
box_colors = [plt.cm.viridis(norm_v(v)) for v in means]

bp = ax.boxplot(box_data, patch_artist=True, vert=True,
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.5),
                flierprops=dict(marker='.', markersize=4, alpha=0.5))
for patch, col in zip(bp['boxes'], box_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.88)
for flier, col in zip(bp['fliers'], box_colors):
    flier.set_markerfacecolor(col)
    flier.set_markeredgecolor(col)

ax.set_xticks(range(1, len(box_names)+1))
ax.set_xticklabels(box_names, fontsize=10)
ax.set_ylabel('Per-class ECE')
ax.set_title('154-Class · Per-Class ECE Distribution\n'
             '(each point = ECE for one gloss class; Viridis = mean ECE)',
             fontsize=12, fontweight='bold', pad=12)
ax.grid(axis='y', alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm_v)
sm.set_array([])
plt.colorbar(sm, ax=ax, fraction=0.025, label='Mean ECE (Viridis)')

for i, (m, d) in enumerate(zip(box_names, box_data), start=1):
    ax.text(i, ax.get_ylim()[1]*0.97,
            f'μ={np.mean(d):.3f}',
            ha='center', fontsize=8, color=TOL.get(m,'grey'), fontweight='bold')

ax.text(0.01, -0.13,
        'Note. Each box shows the ECE distribution across individual gloss classes. '
        'Wider spread = inconsistent calibration across classes. '
        'μ = mean per-class ECE.',
        transform=ax.transAxes, fontsize=7.5, color='#555555')
plt.tight_layout()
save_fig(fig, 'CAL4_per_class_ece_boxplot.png')

In [ ]:
# ── Cell 5: Summary table + manifest ─────────────────────────────────────
summary_rows = []
for m in MODEL_ORDER:
    if m not in results:
        summary_rows.append({'Model': m, 'ECE (pooled)': '—',
                             'Macro Brier': '—', 'Mean per-class ECE': '—',
                             'Calibration quality': 'pending'})
        continue
    r   = results[m]
    ece = r['ece_pooled']
    quality = ('Excellent' if ece < 0.02 else
               'Good'      if ece < 0.05 else
               'Moderate'  if ece < 0.10 else
               'Poor')
    summary_rows.append({
        'Model':               m,
        'ECE (pooled)':        f'{ece:.4f}',
        'Macro Brier':         f"{r['macro_brier']:.4f}",
        'Mean per-class ECE':  f"{np.mean(r['ece_per_class']):.4f}",
        'Calibration quality': quality,
    })

df_summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, 'CAL5_calibration_summary.csv')
df_summary.to_csv(summary_path, index=False)

print('\n154-CLASS — CALIBRATION SUMMARY')
print('='*68)
print(df_summary.to_string(index=False))
print()
print('Calibration quality thresholds:')
print('  Excellent : ECE < 0.02')
print('  Good      : ECE < 0.05')
print('  Moderate  : ECE < 0.10')
print('  Poor      : ECE ≥ 0.10')

print('\n' + '='*68)
print('  OUTPUTS SAVED → results/end_point/')
print('='*68)
cal_files = sorted(f for f in os.listdir(OUT_DIR) if f.startswith('CAL'))
for fname in cal_files:
    kb = os.path.getsize(os.path.join(OUT_DIR, fname)) // 1024
    print(f'  ✅  {fname:<55} {kb:>5} KB')
print()
print('  CAL1 — All 4 models on one reliability diagram')
print('  CAL2 — Individual 2×2 grid with confidence histograms')
print('  CAL3 — ECE + Brier score comparison bars')
print('  CAL4 — Per-class ECE distribution box plots')
print('  CAL5 — Calibration summary CSV')